In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler --quiet
import awswrangler as wr
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET   = 'gold-lstm-forecast'
BRONZE_PATH = f's3://{S3_BUCKET}/bronze/xauusd_daily/raw'
SILVER_PATH = f's3://{S3_BUCKET}/silver'
print(f'Bronze : {BRONZE_PATH}')
print(f'Silver : {SILVER_PATH}')

In [ ]:
# Cell 2 · Load Bronze Data
def read_bronze(filename):
    path = f'{BRONZE_PATH}/{filename}'
    df   = wr.s3.read_csv(path=path)
    if len(df.columns) == 1:
        df = wr.s3.read_csv(path=path, sep=';')
    return df

df_1d = read_bronze('XAU_1d_data.csv')
df_1w = read_bronze('XAU_1w_data.csv')
df_1m = read_bronze('XAU_1Month_data.csv')

print('=== Bronze Raw Shape ===')
for name, df in [('1D',df_1d),('1W',df_1w),('1M',df_1m)]:
    print(f'  {name} : {df.shape}  columns: {df.columns.tolist()}')

In [ ]:
# Cell 3 · Clean Columns & Fix Date
def clean_df(df, tf_name):
    df = df.copy()
    col_map = {}
    for c in df.columns:
        cl = c.strip().lower()
        if 'date' in cl or 'time' in cl: col_map[c] = 'date'
        elif cl == 'open':               col_map[c] = 'open'
        elif cl == 'high':               col_map[c] = 'high'
        elif cl == 'low':                col_map[c] = 'low'
        elif cl == 'close':              col_map[c] = 'close'
        elif 'vol' in cl:                col_map[c] = 'volume'
    df = df.rename(columns=col_map)
    df['date'] = df['date'].astype(str).str.replace('.', '-', regex=False).str[:10]
    df['date'] = pd.to_datetime(df['date'], errors='coerce')
    bad_dates = df['date'].isna().sum()
    if bad_dates > 0:
        print(f'  [{tf_name}] dropped {bad_dates} unparseable dates')
        df = df.dropna(subset=['date'])
    for col in ['open','high','low','close','volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.sort_values('date').reset_index(drop=True)
    print(f'  [{tf_name}] {len(df):,} rows | {df["date"].min().date()} -> {df["date"].max().date()}')
    return df

print('=== Cleaning ===')
df_1d = clean_df(df_1d, '1D')
df_1w = clean_df(df_1w, '1W')
df_1m = clean_df(df_1m, '1M')

In [ ]:
# Cell 4 · Check Missing Values
print('=== Missing Values ===')
for name, df in [('1D',df_1d),('1W',df_1w),('1M',df_1m)]:
    nulls = df.isnull().sum()
    total = nulls.sum()
    print(f'  [{name}] total missing: {total}')
    if total > 0:
        print(nulls[nulls > 0])

In [ ]:
# Cell 5 · Resample Multi-timeframe to Daily
df_1w_daily = df_1w.set_index('date').resample('1D').ffill().reset_index()
df_1w_daily.columns = ['date'] + [f'{c}_1w' for c in df_1w_daily.columns[1:]]

df_1m_daily = df_1m.set_index('date').resample('1D').ffill().reset_index()
df_1m_daily.columns = ['date'] + [f'{c}_1m' for c in df_1m_daily.columns[1:]]

print('Resample done')
print(f'  1W daily : {df_1w_daily.shape}')
print(f'  1M daily : {df_1m_daily.shape}')

In [ ]:
# Cell 6 · Merge All Timeframes
df = df_1d.copy()
df = df.merge(df_1w_daily, on='date', how='left')
df = df.merge(df_1m_daily, on='date', how='left')
df = df.sort_values('date').reset_index(drop=True)
df = df.ffill()

before = len(df)
df = df.dropna().reset_index(drop=True)
after  = len(df)

print(f'=== After Merge ===')
print(f'  Shape  : {df.shape}')
print(f'  Dropped: {before - after} rows (ช่วงต้นที่ขาดข้อมูล)')
print(f'  Date   : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'  Columns: {df.columns.tolist()}')

In [ ]:
# Cell 7 · Final Validation
print('=== Silver Validation ===')
print(f'  Shape        : {df.shape}')
print(f'  Date range   : {df["date"].min().date()} -> {df["date"].max().date()}')
print(f'  Missing vals : {df.isnull().sum().sum()}')
print(f'  Duplicates   : {df.duplicated(subset=["date"]).sum()}')
print(f'  Chronological: {df["date"].is_monotonic_increasing}')
print()
print(df[['date','open','high','low','close','volume']].head(3).to_string(index=False))

In [ ]:
# Cell 8 · Save to Silver Layer
silver_path = f'{SILVER_PATH}/xauusd_daily_clean.parquet'
wr.s3.to_parquet(df=df, path=silver_path, index=False)
print(f'Silver saved -> {silver_path}')
print(f'Shape : {df.shape}')

In [ ]:
# Cell 9 · Verify Read-back
verify = wr.s3.read_parquet(path=silver_path)
print('=== Verify Silver ===')
print(f'  Shape      : {verify.shape}')
print(f'  Date range : {verify["date"].min()} -> {verify["date"].max()}')
print(f'  Missing    : {verify.isnull().sum().sum()}')
print()
print('Notebook 02 DONE')
print('Next -> 03_Feature_Engineering.ipynb')